# 07. Split-Apply-Combine Groupby Operations: Beginner Guide

### 📌 Overview
Master **07. Split-Apply-Combine Groupby Operations: Beginner Guide** with concise, zero-fluff bullet points and executable code on real Fintech records ([raw_transactions.csv](file:///data/raw_transactions.csv)).

### 📚 Key Concepts Covered in this Notebook:
- **Grouping**: Covers `df.groupby()`.
- **Multiple Aggregations**: Covers `.agg()` with multiple metrics.
- **Named Aggregations**: Covers named aggregation tuples (`total_amt=('transaction_amount', 'sum')`).
- **Group Transformation**: Covers `.transform()`.
- **Group Filtering**: Covers `.filter()`.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import sqlite3
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(2))

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns
  transaction_id customer_id merchant_id  transaction_amount card_type  \
0       TX109326      C55082       M3549              607.78      Visa   
1       TX106376      C76616       M3068             1819.11      Visa   

  transaction_status device_type  account_age_months     transaction_date  \
0           Reversed      Mobile                   8  2026-02-17 08:28:57   
1            Pending         POS                  28          03-Jan-2025   

  region  is_fraud  
0  North         0  
1   West         1  


### 🔹 Split-Apply-Combine Grouping: `DataFrame.groupby()`
- **What it does:** Splits the DataFrame into groups based on specified grouping keys/columns for subsequent aggregation, transformation, or filtering.
- **Syntax:** `DataFrame.groupby()`
- **Key Note:** Grouping is lazy: calling `.groupby()` does not compute results until an aggregation (`.sum()`, `.mean()`, `.agg()`) is invoked.
- **Dataset Application & Code Demonstration:** Applies Split-Apply-Combine Grouping on fintech records using columns `region` to demonstrate real-world execution.


In [2]:
grouped_region = df.groupby('region')

### 🔹 Multi-Metric Aggregation: `DataFrameGroupBy.agg()`
- **What it does:** Aggregates grouped data using one or more operations across specified columns.
- **Syntax:** `DataFrameGroupBy.agg()`
- **Key Note:** Passing a dictionary mapping column names to aggregation lists allows distinct mathematical summaries per column in a single pass.
- **Dataset Application & Code Demonstration:** Applies Multi-Metric Aggregation on fintech records using columns `is_fraud`, `region`, `transaction_amount` to demonstrate real-world execution.


In [3]:
regional_summary = df.groupby('region').agg({
    'transaction_amount': ['sum', 'mean', 'count'],
    'is_fraud': 'mean'
})
print('Regional Multi-Metric Summary:\n', regional_summary)

Regional Multi-Metric Summary:
         transaction_amount                     is_fraud
                       sum         mean count      mean
region                                                 
 East             71215.44   975.553973    73  0.077922
 North            77668.31  1078.726528    72  0.081081
 South            93212.31  1096.615412    85  0.141176
 West             71931.40   899.142500    80  0.048193
East            3483441.29   994.700540  3502  0.098910
North           3374657.98  1002.274422  3367  0.113874
South           3362996.71  1005.981666  3343  0.105189
West            3458096.29  1009.073910  3427  0.112304
east              93389.78  1111.783095    84  0.160920
north             62709.34  1100.163860    57  0.189655
south             95901.77  1089.792841    88  0.098901
west              81714.88  1119.381918    73  0.118421


### 🔹 Named Aggregations: `DataFrameGroupBy.agg(**kwargs)`
- **What it does:** Computes aggregations while assigning clean, explicit custom column names to the output DataFrame, avoiding ugly MultiIndex column tuples.
- **Syntax:** `DataFrameGroupBy.agg(**kwargs)`
- **Key Note:** Named aggregation prevents nested MultiIndex columns and produces ready-to-export tabular reporting summaries.
- **Dataset Application & Code Demonstration:** Applies Named Aggregations on fintech records using columns `is_fraud`, `region`, `transaction_amount`, `transaction_id` to demonstrate real-world execution.


In [4]:
clean_named_agg = df.groupby('region').agg(
    total_spend=('transaction_amount', 'sum'),
    avg_spend=('transaction_amount', 'mean'),
    fraud_rate=('is_fraud', 'mean'),
    tx_count=('transaction_id', 'count')
)
print('Clean Named Aggregations Table:\n', clean_named_agg.round(3))

Clean Named Aggregations Table:
          total_spend  avg_spend  fraud_rate  tx_count
region                                               
 East       71215.44    975.554       0.078        77
 North      77668.31   1078.727       0.081        74
 South      93212.31   1096.615       0.141        85
 West       71931.40    899.142       0.048        83
East      3483441.29    994.701       0.099      3670
North     3374657.98   1002.274       0.114      3539
South     3362996.71   1005.982       0.105      3527
West      3458096.29   1009.074       0.112      3633
east        93389.78   1111.783       0.161        87
north       62709.34   1100.164       0.190        58
south       95901.77   1089.793       0.099        91
west        81714.88   1119.382       0.118        76


### 🔹 Group Transformation: `DataFrameGroupBy.transform()`
- **What it does:** Computes grouped aggregation statistics but broadcasts the results back to the original DataFrame's shape, preserving row alignment.
- **Syntax:** `DataFrameGroupBy.transform()`
- **Key Note:** Because `.transform()` returns an array with the exact same length as the original dataframe, it can be directly assigned as a new column.
- **Dataset Application & Code Demonstration:** Applies Group Transformation on fintech records using columns `customer_id`, `transaction_amount`, `transaction_id` to demonstrate real-world execution.


In [5]:
df_sub = df.dropna(subset=['transaction_amount']).head(100).copy()
df_sub['cust_avg_amt'] = df_sub.groupby('customer_id')['transaction_amount'].transform('mean')
df_sub['amt_ratio'] = df_sub['transaction_amount'] / df_sub['cust_avg_amt']
print(df_sub[['transaction_id', 'customer_id', 'transaction_amount', 'cust_avg_amt', 'amt_ratio']].head(5))

  transaction_id customer_id  transaction_amount  cust_avg_amt  amt_ratio
0       TX109326      C55082              607.78        607.78        1.0
1       TX106376      C76616             1819.11       1819.11        1.0
2       TX103301      C65296               64.08         64.08        1.0
3       TX110701      C42098             1025.73       1025.73        1.0
4       TX103284      C97782              772.74        772.74        1.0


### 🔹 Group Filtering: `DataFrameGroupBy.filter()`
- **What it does:** Filters entire groups in or out of the DataFrame based on a group-level boolean predicate function.
- **Syntax:** `DataFrameGroupBy.filter()`
- **Key Note:** Unlike standard row filtering, `.filter()` either retains or discards the entire group of rows collectively.
- **Dataset Application & Code Demonstration:** Applies Group Filtering on fintech records using columns `customer_id`, `transaction_amount`, `transaction_id` to demonstrate real-world execution.


In [6]:
high_vol_cust = df.groupby('customer_id').filter(lambda g: len(g) >= 5)
print(f'Transactions from High-Volume Customers: {len(high_vol_cust)}')
print(high_vol_cust[['transaction_id', 'customer_id', 'transaction_amount']].head(3))

Transactions from High-Volume Customers: 14824
  transaction_id customer_id  transaction_amount
0       TX109326      C55082              607.78
1       TX106376      C76616             1819.11
2       TX103301      C65296               64.08


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Identifying Outlier Transactions with Group Z-Scores
- **Objective:** Q1: Identifying Outlier Transactions with Group Z-Scores
- **Approach:** Flag transactions that are 3 standard deviations above the customer's average spend.
- **Syntax:** `(df['amt'] - df.groupby('cust')['amt'].transform('mean')) / df.groupby('cust')['amt'].transform('std') > 3`

In [7]:
cust_means = df.groupby('customer_id')['transaction_amount'].transform('mean')
cust_stds = df.groupby('customer_id')['transaction_amount'].transform('std').fillna(1.0)
df_flagged = df.assign(z_score=(df['transaction_amount'] - cust_means) / cust_stds)
outliers = df_flagged[df_flagged['z_score'] > 2.5]
print(f'Found {len(outliers)} statistical outlier transactions!')
print(outliers[['transaction_id', 'customer_id', 'transaction_amount', 'z_score']].head(3))

Found 0 statistical outlier transactions!
Empty DataFrame
Columns: [transaction_id, customer_id, transaction_amount, z_score]
Index: []


### 🔍 Scenario: Multi-Table Fintech GroupBy (Merchant Category Revenue & Risk Aggregation)
- **Objective:** Multi-Table Fintech GroupBy (Merchant Category Revenue & Risk Aggregation)
- **Approach:** Apply built-in transformations and inspect output integrity.

In [8]:
# Multi-Table Aggregation across Transactions and Merchants
data_dir = 'data' if os.path.exists('data/merchants.csv') else '../data'
df_merch = pd.read_csv(f'{data_dir}/merchants.csv')

tx_merged = df.merge(df_merch[['merchant_id', 'category', 'interchange_fee_pct', 'risk_rating']], on='merchant_id', how='left')
tx_merged['fee_income'] = tx_merged['transaction_amount'] * tx_merged['interchange_fee_pct']

category_risk_pivot = tx_merged.groupby(['category', 'risk_rating']).agg(
    tx_count=('transaction_id', 'count'),
    total_volume=('transaction_amount', 'sum'),
    avg_tx_amount=('transaction_amount', 'mean'),
    total_fee_income=('fee_income', 'sum')
).round(2)

print("Multi-Table GroupBy (Merchant Category & Risk Rating):")
print(category_risk_pivot.head(10))


Multi-Table GroupBy (Merchant Category & Risk Rating):
                                       tx_count  total_volume  avg_tx_amount  \
category                  risk_rating                                          
Crypto & Digital Assets   Extreme          1577    1497653.00         996.44   
                          High              102      87770.12         886.57   
                          Low                33      25439.27         877.22   
                          Moderate          106      95163.52         923.92   
E-Commerce & Marketplaces Extreme            47      49935.39        1085.55   
                          High               65      47857.75         839.61   
                          Moderate         1519    1416045.62         992.32   
Electronics & Computers   High               25      26692.57        1112.19   
                          Low                28      28834.89        1067.96   
                          Moderate         1435    1357255.87    